In [4]:
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

DATA_PATH = (
    PROJECT_ROOT
    / "Data"
    / "processed"
    / "heatwave_prediction.parquet1"
)

heatwave_df = pd.read_parquet(DATA_PATH)

print("Shape:", heatwave_df.shape)


Shape: (2056547, 22)


In [5]:
heatwave_df.columns

Index(['date', 'latitude', 'longitude', 'month', 'day_of_year', 'month_sin',
       'month_cos', 'temp_lag_1', 'temp_lag_2', 'temp_lag_3', 'temp_lag_7',
       'temp_rolling_mean_3', 'temp_rolling_mean_7', 'temp_rolling_std_7',
       'temperature_change_1d', 'temperature_change_3d', 'anomaly_lag_1',
       'extreme_heat_streak', 'p90_temperature', 'p95_temperature',
       'p99_temperature', 'target_heatwave'],
      dtype='str')

In [6]:
train_df = heatwave_df[
    heatwave_df["date"].dt.year <= 2022
].copy()

val_df = heatwave_df[
    heatwave_df["date"].dt.year == 2023
].copy()

test_df = heatwave_df[
    heatwave_df["date"].dt.year >= 2024
].copy()

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

Train: (1669175, 22)
Validation: (129185, 22)
Test: (258187, 22)


In [7]:
print("TRAIN")
print(train_df["target_heatwave"].value_counts(normalize=True) * 100)

print("\nVALIDATION")
print(val_df["target_heatwave"].value_counts(normalize=True) * 100)

print("\nTEST")
print(test_df["target_heatwave"].value_counts(normalize=True) * 100)

TRAIN
target_heatwave
0    94.911199
1     5.088801
Name: proportion, dtype: Float64

VALIDATION
target_heatwave
0    95.804466
1     4.195534
Name: proportion, dtype: Float64

TEST
target_heatwave
0    94.980382
1     5.019618
Name: proportion, dtype: Float64


In [9]:
features = [
    "latitude",
    "longitude",

    "month",
    "day_of_year",
    "month_sin",
    "month_cos",

    "temp_lag_1",
    "temp_lag_2",
    "temp_lag_3",
    "temp_lag_7",

    "temp_rolling_mean_3",
    "temp_rolling_mean_7",
    "temp_rolling_std_7",

    "temperature_change_1d",
    "temperature_change_3d",

    "anomaly_lag_1",

    "extreme_heat_streak"
] 
classification_features = features + [
    "p90_temperature",
    "p95_temperature",
    "p99_temperature"
]

In [10]:
X_train = train_df[classification_features]
y_train = train_df["target_heatwave"]

X_val = val_df[classification_features]
y_val = val_df["target_heatwave"]

X_test = test_df[classification_features]
y_test = test_df["target_heatwave"]

In [11]:
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_val:", X_val.shape)
print("y_val:", y_val.shape)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: (1669175, 20)
y_train: (1669175,)
X_val: (129185, 20)
y_val: (129185,)
X_test: (258187, 20)
y_test: (258187,)


In [12]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

baseline_model = Pipeline([
    ("scaler", StandardScaler()),
    (
        "model",
        LogisticRegression(
            class_weight="balanced",
            max_iter=1000,
            random_state=42
        )
    )
])

baseline_model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('scaler', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[float64](2,)","[0.,1.]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](20,)","['latitude','longitude','month',...,'p90_temperature','p95_temperature', 'p99_temperature']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,20
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True


In [13]:
y_pred = baseline_model.predict(X_val)
y_prob = baseline_model.predict_proba(X_val)[:, 1]

In [14]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score
)

print(classification_report(
    y_val,
    y_pred,
    digits=4
))

print("ROC-AUC:", roc_auc_score(y_val, y_prob))

print(
    "PR-AUC:",
    average_precision_score(y_val, y_prob)
)

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, y_pred))

              precision    recall  f1-score   support

         0.0     0.9959    0.9105    0.9512    123765
         1.0     0.3088    0.9137    0.4616      5420

    accuracy                         0.9106    129185
   macro avg     0.6524    0.9121    0.7064    129185
weighted avg     0.9670    0.9106    0.9307    129185

ROC-AUC: 0.974623328671779
PR-AUC: 0.7019592845490993

Confusion Matrix:
[[112683  11082]
 [   468   4952]]


In [15]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="aucpr",
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)
scale_pos_weight = (
    (y_train == 0).sum()
    / (y_train == 1).sum()
)

print("scale_pos_weight:", scale_pos_weight)

scale_pos_weight: 18.650993042229313


In [16]:
xgb_model.set_params(
    scale_pos_weight=scale_pos_weight
)
xgb_model.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
)

,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",'aucpr'
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [17]:
xgb_pred = xgb_model.predict(X_val)

xgb_prob = xgb_model.predict_proba(X_val)[:, 1]
print(classification_report(
    y_val,
    xgb_pred,
    digits=4
))

print("ROC-AUC:", roc_auc_score(y_val, xgb_prob))

print(
    "PR-AUC:",
    average_precision_score(y_val, xgb_prob)
)

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, xgb_pred))

              precision    recall  f1-score   support

         0.0     0.9961    0.9212    0.9572    123765
         1.0     0.3378    0.9175    0.4938      5420

    accuracy                         0.9211    129185
   macro avg     0.6669    0.9194    0.7255    129185
weighted avg     0.9685    0.9211    0.9378    129185

ROC-AUC: 0.9796337668563936
PR-AUC: 0.7982977422451774

Confusion Matrix:
[[114015   9750]
 [   447   4973]]


In [18]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score
)

threshold_results = []

for threshold in np.arange(0.10, 0.91, 0.05):

    pred = (xgb_prob >= threshold).astype(int)

    threshold_results.append({
        "threshold": threshold,
        "precision": precision_score(y_val, pred),
        "recall": recall_score(y_val, pred),
        "f1": f1_score(y_val, pred)
    })

threshold_df = pd.DataFrame(threshold_results)

threshold_df

,threshold,precision,recall,f1
0,0.10,0.194995,0.990590,0.325848
1,0.15,0.213477,0.984871,0.350896
2,0.20,0.229589,0.974908,0.371655
3,0.25,0.244835,0.968635,0.390872
4,0.30,0.259156,0.958303,0.407981
5,0.35,0.275248,0.952399,0.427070
6,0.40,0.292998,0.942620,0.447040
7,0.45,0.313812,0.932288,0.469566
8,0.50,0.337771,0.917528,0.493770
9,0.55,0.362673,0.897048,0.516520


In [24]:
best_row = threshold_df.loc[threshold_df["f1"].idxmax()]

best_threshold = best_row["threshold"]

print(best_row)

threshold    0.900000
precision    0.855692
recall       0.668450
f1           0.750570
Name: 16, dtype: float64


In [25]:
test_prob = xgb_model.predict_proba(X_test)[:, 1]

test_pred = (
    test_prob >= best_threshold
).astype(int)

print(classification_report(y_test, test_pred))

print("ROC-AUC:", roc_auc_score(y_test, test_prob))
print("PR-AUC:", average_precision_score(y_test, test_prob))

print("Confusion Matrix:")
print(confusion_matrix(y_test, test_pred))

              precision    recall  f1-score   support

         0.0       0.98      0.99      0.99    245227
         1.0       0.87      0.71      0.78     12960

    accuracy                           0.98    258187
   macro avg       0.93      0.85      0.88    258187
weighted avg       0.98      0.98      0.98    258187

ROC-AUC: 0.9818468643149831
PR-AUC: 0.8355692972472051
Confusion Matrix:
[[243879   1348]
 [  3806   9154]]


In [26]:
heatwave_df.columns

Index(['date', 'latitude', 'longitude', 'month', 'day_of_year', 'month_sin',
       'month_cos', 'temp_lag_1', 'temp_lag_2', 'temp_lag_3', 'temp_lag_7',
       'temp_rolling_mean_3', 'temp_rolling_mean_7', 'temp_rolling_std_7',
       'temperature_change_1d', 'temperature_change_3d', 'anomaly_lag_1',
       'extreme_heat_streak', 'p90_temperature', 'p95_temperature',
       'p99_temperature', 'target_heatwave'],
      dtype='str')

In [27]:
from sklearn.metrics import precision_score, recall_score, f1_score

val_prob = xgb_model.predict_proba(X_val)[:, 1]

threshold_results = []

for threshold in np.arange(0.10, 0.96, 0.01):

    pred = (val_prob >= threshold).astype(int)

    threshold_results.append({
        "threshold": round(threshold, 2),
        "precision": precision_score(y_val, pred),
        "recall": recall_score(y_val, pred),
        "f1": f1_score(y_val, pred)
    })

threshold_df = pd.DataFrame(threshold_results)

best_row = threshold_df.loc[threshold_df["f1"].idxmax()]

print(best_row)

threshold    0.900000
precision    0.855692
recall       0.668450
f1           0.750570
Name: 80, dtype: float64


In [28]:
test_prob = xgb_model.predict_proba(X_test)[:, 1]

test_pred = (test_prob >= 0.90).astype(int)

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score
)

print(classification_report(y_test, test_pred))

print("ROC-AUC:", roc_auc_score(y_test, test_prob))
print("PR-AUC:", average_precision_score(y_test, test_prob))

print("Confusion Matrix:")
print(confusion_matrix(y_test, test_pred))

              precision    recall  f1-score   support

         0.0       0.98      0.99      0.99    245227
         1.0       0.87      0.71      0.78     12960

    accuracy                           0.98    258187
   macro avg       0.93      0.85      0.88    258187
weighted avg       0.98      0.98      0.98    258187

ROC-AUC: 0.9818468643149831
PR-AUC: 0.8355692972472051
Confusion Matrix:
[[243879   1348]
 [  3806   9154]]


In [29]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score
)

# Get probabilities
test_prob = xgb_model.predict_proba(X_test)[:, 1]

# IMPORTANT: threshold = 0.90
test_pred_90 = (test_prob >= 0.90).astype(int)

print("Number predicted as heatwave:", test_pred_90.sum())
print("Actual heatwave cases:", y_test.sum())

print("\nClassification Report:")
print(classification_report(y_test, test_pred_90))

print("ROC-AUC:", roc_auc_score(y_test, test_prob))
print("PR-AUC:", average_precision_score(y_test, test_prob))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, test_pred_90))

Number predicted as heatwave: 10502
Actual heatwave cases: 12960

Classification Report:
              precision    recall  f1-score   support

         0.0       0.98      0.99      0.99    245227
         1.0       0.87      0.71      0.78     12960

    accuracy                           0.98    258187
   macro avg       0.93      0.85      0.88    258187
weighted avg       0.98      0.98      0.98    258187

ROC-AUC: 0.9818468643149831
PR-AUC: 0.8355692972472051

Confusion Matrix:
[[243879   1348]
 [  3806   9154]]


In [30]:
print("Min probability:", test_prob.min())
print("Max probability:", test_prob.max())

print("\nProbability percentiles:")
print(np.percentile(
    test_prob,
    [50, 75, 90, 95, 97, 98, 99, 99.5]
))

print("\nPredictions:")
for t in [0.5, 0.7, 0.8, 0.9, 0.95]:
    pred = (test_prob >= t).astype(int)
    print(
        f"Threshold {t}: "
        f"{pred.sum()} predicted heatwaves"
    )

Min probability: 7.729995e-07
Max probability: 0.99984777

Probability percentiles:
[6.28571608e-04 5.90156354e-02 6.03674769e-01 8.20838487e-01
 9.82263585e-01 9.93478830e-01 9.97447796e-01 9.98565068e-01]

Predictions:
Threshold 0.5: 31584 predicted heatwaves
Threshold 0.7: 20239 predicted heatwaves
Threshold 0.8: 14016 predicted heatwaves
Threshold 0.9: 10502 predicted heatwaves
Threshold 0.95: 9628 predicted heatwaves


In [31]:
print(test_prob[:20])

[0.00271442 0.00103563 0.00277988 0.00058326 0.00057415 0.00035352
 0.00047654 0.00045772 0.00060802 0.00021845 0.00035149 0.00096701
 0.00084018 0.00153361 0.00161558 0.00164447 0.0016314  0.00200402
 0.00133827 0.00157846]


In [32]:
print(test_prob[:20])

[0.00271442 0.00103563 0.00277988 0.00058326 0.00057415 0.00035352
 0.00047654 0.00045772 0.00060802 0.00021845 0.00035149 0.00096701
 0.00084018 0.00153361 0.00161558 0.00164447 0.0016314  0.00200402
 0.00133827 0.00157846]


In [33]:
threshold = 0.90

test_pred_90 = (test_prob >= threshold).astype(int)

print("Threshold:", threshold)
print("Predicted heatwaves:", test_pred_90.sum())
print("Actual heatwaves:", y_test.sum())

print("\nClassification Report:")
print(classification_report(y_test, test_pred_90))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, test_pred_90))

Threshold: 0.9
Predicted heatwaves: 10502
Actual heatwaves: 12960

Classification Report:
              precision    recall  f1-score   support

         0.0       0.98      0.99      0.99    245227
         1.0       0.87      0.71      0.78     12960

    accuracy                           0.98    258187
   macro avg       0.93      0.85      0.88    258187
weighted avg       0.98      0.98      0.98    258187


Confusion Matrix:
[[243879   1348]
 [  3806   9154]]


In [3]:
import pandas as pd
import numpy as np

df = pd.read_parquet(
    "../Data/processed/heatwave_prediction.parquet1"
)

print(df.shape)
df.head()

(2056547, 22)


,date,latitude,longitude,month,day_of_year,month_sin,month_cos,temp_lag_1,temp_lag_2,temp_lag_3,...,temp_rolling_mean_7,temp_rolling_std_7,temperature_change_1d,temperature_change_3d,anomaly_lag_1,extreme_heat_streak,p90_temperature,p95_temperature,p99_temperature,target_heatwave
0,2010-01-08,8.5,73.5,1,8,0.137185,0.990545,32.259998,32.270000,32.020000,...,32.315714,0.331202,-0.010002,0.239998,-0.417439,0,34.336103,34.857802,35.694044,0
1,2010-01-09,8.5,73.5,1,9,0.154204,0.988039,32.320000,32.259998,32.270000,...,32.237143,0.226474,0.060001,0.049999,-0.357437,0,34.336103,34.857802,35.694044,0
2,2010-01-10,8.5,73.5,1,10,0.171177,0.985240,32.669998,32.320000,32.259998,...,32.258571,0.262006,0.349998,0.410000,-0.007439,0,34.336103,34.857802,35.694044,0
3,2010-01-11,8.5,73.5,1,11,0.188099,0.982150,29.540001,32.669998,32.320000,...,31.848571,1.049037,-3.129997,-2.779999,-3.137436,0,34.336103,34.857802,35.694044,0
4,2010-01-12,8.5,73.5,1,12,0.204966,0.978769,32.250000,29.540001,32.669998,...,31.904285,1.060044,2.709999,-0.419998,-0.427437,0,34.336103,34.857802,35.694044,0


In [4]:
target = "target_heatwave"

features = [
    col for col in df.columns
    if col not in ["date", target]
]

X = df[features]
y = df[target]

print("Features:", len(features))
print("X shape:", X.shape)
print("y shape:", y.shape)

Features: 20
X shape: (2056547, 20)
y shape: (2056547,)


In [5]:
df["date"] = pd.to_datetime(df["date"])

train_mask = df["date"].dt.year <= 2022
val_mask = df["date"].dt.year == 2023
test_mask = df["date"].dt.year >= 2024

X_train = df.loc[train_mask, features]
y_train = df.loc[train_mask, target]

X_val = df.loc[val_mask, features]
y_val = df.loc[val_mask, target]

X_test = df.loc[test_mask, features]
y_test = df.loc[test_mask, target]

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

Train: (1669175, 20)
Validation: (129185, 20)
Test: (258187, 20)


In [6]:
from xgboost import XGBClassifier

scale_pos_weight = (
    (y_train == 0).sum() /
    (y_train == 1).sum()
)

xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="aucpr",
    tree_method="hist",
    random_state=42,
    n_jobs=-1,
    scale_pos_weight=scale_pos_weight
)

xgb_model.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
)

,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",'aucpr'
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [7]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score
)

val_prob = xgb_model.predict_proba(X_val)[:, 1]

val_pred = (val_prob >= 0.5).astype(int)

print(classification_report(y_val, val_pred))

print("ROC-AUC:", roc_auc_score(y_val, val_prob))
print("PR-AUC:", average_precision_score(y_val, val_prob))

print("Confusion Matrix:")
print(confusion_matrix(y_val, val_pred))

              precision    recall  f1-score   support

         0.0       1.00      0.92      0.96    123765
         1.0       0.34      0.92      0.49      5420

    accuracy                           0.92    129185
   macro avg       0.67      0.92      0.73    129185
weighted avg       0.97      0.92      0.94    129185

ROC-AUC: 0.9796337668563936
PR-AUC: 0.7982977422451774
Confusion Matrix:
[[114015   9750]
 [   447   4973]]


In [8]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score
)

val_prob = xgb_model.predict_proba(X_val)[:, 1]

val_pred = (val_prob >= 0.5).astype(int)

print(classification_report(y_val, val_pred))

print("ROC-AUC:", roc_auc_score(y_val, val_prob))
print("PR-AUC:", average_precision_score(y_val, val_prob))

print("Confusion Matrix:")
print(confusion_matrix(y_val, val_pred))

              precision    recall  f1-score   support

         0.0       1.00      0.92      0.96    123765
         1.0       0.34      0.92      0.49      5420

    accuracy                           0.92    129185
   macro avg       0.67      0.92      0.73    129185
weighted avg       0.97      0.92      0.94    129185

ROC-AUC: 0.9796337668563936
PR-AUC: 0.7982977422451774
Confusion Matrix:
[[114015   9750]
 [   447   4973]]


In [9]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score
)

# Get probabilities
test_prob = xgb_model.predict_proba(X_test)[:, 1]

# IMPORTANT: threshold = 0.90
test_pred_90 = (test_prob >= 0.90).astype(int)

print("Number predicted as heatwave:", test_pred_90.sum())
print("Actual heatwave cases:", y_test.sum())

print("\nClassification Report:")
print(classification_report(y_test, test_pred_90))

print("ROC-AUC:", roc_auc_score(y_test, test_prob))
print("PR-AUC:", average_precision_score(y_test, test_prob))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, test_pred_90))

Number predicted as heatwave: 10502
Actual heatwave cases: 12960

Classification Report:
              precision    recall  f1-score   support

         0.0       0.98      0.99      0.99    245227
         1.0       0.87      0.71      0.78     12960

    accuracy                           0.98    258187
   macro avg       0.93      0.85      0.88    258187
weighted avg       0.98      0.98      0.98    258187

ROC-AUC: 0.9818468643149831
PR-AUC: 0.8355692972472051

Confusion Matrix:
[[243879   1348]
 [  3806   9154]]


In [10]:
from sklearn.metrics import precision_score, recall_score, f1_score

val_prob = xgb_model.predict_proba(X_val)[:, 1]

threshold_results = []

for threshold in np.arange(0.10, 0.96, 0.01):

    pred = (val_prob >= threshold).astype(int)

    threshold_results.append({
        "threshold": round(threshold, 2),
        "precision": precision_score(y_val, pred),
        "recall": recall_score(y_val, pred),
        "f1": f1_score(y_val, pred)
    })

threshold_df = pd.DataFrame(threshold_results)

best_row = threshold_df.loc[threshold_df["f1"].idxmax()]

print(best_row)

threshold    0.900000
precision    0.855692
recall       0.668450
f1           0.750570
Name: 80, dtype: float64


In [11]:
import pandas as pd

importance = pd.Series(
    xgb_model.feature_importances_,
    index=features
).sort_values(ascending=False)

print(importance)

extreme_heat_streak      0.447639
temp_lag_1               0.130903
p90_temperature          0.084762
month                    0.056393
month_cos                0.053877
p95_temperature          0.040427
temp_rolling_mean_3      0.036312
longitude                0.025810
p99_temperature          0.023598
day_of_year              0.022479
latitude                 0.014899
temp_rolling_mean_7      0.013610
month_sin                0.013538
anomaly_lag_1            0.007951
temp_lag_7               0.006471
temperature_change_1d    0.005783
temp_lag_3               0.004481
temperature_change_3d    0.003846
temp_lag_2               0.003693
temp_rolling_std_7       0.003530
dtype: float32


In [13]:
import joblib

joblib.dump(
    xgb_model,
    "../models/heatwave_xgb.joblib"
)

joblib.dump(
    {
        "features": features,
        "threshold": 0.90
    },
    "../models/heatwave_metadata.joblib"
)

['../models/heatwave_metadata.joblib']